# Modify the fixed slit notebook to work on actual LEGGOS data, using bells and whistles froM Brian Welch
jrigby, Jan 2026.  Original version from B. Welch  
Feb 2026:  Modified to run on multiple processors, which is a lot faster (but sometimes crashes the computer).
Followed [example 2](here https://jwst-pipeline.readthedocs.io/en/1.20.0/jwst/user_documentation/running_pipeline_python.html#multiprocessing)

Regression testing, upgrading from pipeline v1.20.2 to v2.0.1.  The main changes are that NSClean gets 
applied earlier, and there's now picture frame correction.

In [1]:
import os
# 1) where the jwst pipeline config files are located
home = "/Users/jrrigby1/Ref_files/"

# STScI helpdesk says these os commands need to come BEFORE jwst pipeline packages are imported
os.environ["CRDS_PATH"] = home + "crds_cache/jwst_ops"
os.environ["CRDS_SERVER_URL"] = "https://jwst-crds.stsci.edu"
#os.environ["CRDS_CONTEXT"] =   'jwst_1464.pmap'     # for  pipeline v1.20.2, Fall 2025.  THIS WORKED
os.environ["CRDS_CONTEXT"] =   'jwst_1535.pmap'      # For pipeline v2.0.1 , Spring 2026

In [2]:
print("Computer has this many CPUs", int(os.cpu_count()))
cores2use = 4  # Ran out of memory 3/5/2026 with N=5.  If rate files are really big (>~1GB, then easy to run out of memory)
print("Will use", cores2use, "cores")

Computer has this many CPUs 14
Will use 4 cores


In [3]:
# Avoid re-running parts that finished  *** MODIFY AS NEEDED
#pipev = '1.20.2'
pipev = '2.0.1'  # Which version of the pipeline are we running
RUN_LEVEL0  = True
RUN_LEVEL1  = True
RUN_LEVEL2  = True
RUN_LEVEL3  = True
RUN_EXTRACT = True

In [4]:
import numpy as np
import glob
from multiprocessing.pool import Pool
from os.path import basename, dirname, normpath
import matplotlib.pyplot as plt
import pandas
import re
from jrr.jrjwst import writel3asn, extract1D_SB  # Tools from Brian Welch, David Law to make associations
from jrr.jrjwst import   wrap_median_combine_level3_nirspecFS # special median combine and extraction for FS backgrounds
from jrr.spec import mark_CaII_Fraunhofer_lines
from jrr.util import gethead
from jrmulti import run_jwst_det1   # If running mulitprocess in a Jupyter notebook, the function fed to starmap must be a .py file
from jrmulti import run_jwst_spec2  # If running mulitprocess in a Jupyter notebook, the function fed to starmap must be a .py file
from jrmulti import  run_jwst_custom_extraction
from collections import defaultdict
plt.rcParams["figure.figsize"] = (9,4)
import warnings
warnings.filterwarnings('ignore') 

In [5]:
import json
from astropy.io import fits
from astropy.utils.data import download_file
import astropy.units as u
from astropy import wcs
from astropy.wcs import WCS
import matplotlib.pyplot as plt
import matplotlib as mpl

# The calwebb_spec and spec3 pipelines
#from jwst.pipeline import Spec2Pipeline  # This is imported within jrmulti.run_jwst_spec2, docs say to prevent memory leak
from jwst.pipeline import Spec3Pipeline

import jwst
# the level1 pipeline:
#from jwst.pipeline import Detector1Pipeline   # This is imported within jrmulti.run_jwst_det1, docs say to prevent memory leak

# data models
from jwst import datamodels

# association file utilities
from jwst.associations import asn_from_list as afl # Tools for creating association files
from jwst.associations.lib.rules_level2_base import DMSLevel2bBase # Definition of a Lvl2 association file
from jwst.associations.lib.rules_level3_base import DMS_Level3_Base # Definition of a Lvl3 association file

In [6]:
# This had better be 1.20.2, or 2.0X, or STOP
if jwst.__version__ != pipev:
    raise Exception("ERROR, pipeline version", jwst.__version__, 'is not the expectation, pipev')
else : print('Good, pipeline version was as expected:', jwst.__version__)    

Good, pipeline version was as expected: 2.0.1


In [7]:
def poll_spectrum_at_wavelength(wave, spectrum, atwave=3.0):  # find the value of the spectrum at wavelength atwave
    idx = (np.abs(wave - atwave)).argmin()
    return spectrum[idx]

In [8]:
#leggos_in  = '/Volumes/Rawdata/NIRSpec_Fixedslit/CDFS_raw_G140M/'
#leggos_out = '/Users/jrrigby1/SCIENCE/JWST_Data/NIRSpec_Fixedslit/' 
#leggos_in =  '/Volumes/Rawdata/Raw_NIRSpecFS_for_PSF_measurement/G395M/'
#leggos_out = '/Users/jrrigby1/SCIENCE/JWST_Data/NIRSpecFS_reduced_PSF_subsamp_x4/G395M/'
#leggos_in = '/Volumes/Rawdata/NIRSpec_Fixedslit/CDFS_raw_prism/'
#leggos_out = '/Users/jrrigby1/SCIENCE/JWST_Data/NIRSPEC_regression2/'
#leggos_in = '/Volumes/JWST_bkgs_prism/Raw_prism_May2026/'
#leggos_out = '/Volumes/JWST_bkgs_prism/Reduced_prism_May2026/'
leggos_in = '/Users/jrrigby1/SCIENCE/JWST_Data/LEGGOS/SGAS1402m28/Raw_data/'
#leggos_out = '/Users/jrrigby1/SCIENCE/JWST_Data/LEGGOS/SGAS1402m28/Redux_v1.20.2/'
leggos_out = '/Users/jrrigby1/SCIENCE/JWST_Data/LEGGOS/SGAS1402m28/Redux_v2.0.1_adaptivetrace/'
epochdirs = glob.glob(leggos_in + '/*/')

In [9]:
#rerun_these = ['2024-11-22',] # '2023-12-08', '2024-01-06', '2023-05-22', '2023-12-02', '2024-11-21', '2023-12-04', '2023-12-09']
#dirs_to_handle = [a for a in epochdirs if any(b in a for b in rerun_these)]

In [10]:
# Select which epochs to reduce.  All, or subset.  ***** MODIFY THIS *****
#dirs_to_handle = [x for x in epochdirs if 'VID01254006001' in x]
#dirs_to_handle = [leggos_in + x for x in missingL3]
dirs_to_handle = epochdirs
dirs_to_handle

['/Users/jrrigby1/SCIENCE/JWST_Data/LEGGOS/SGAS1402m28/Raw_data/F170LP_G235H/',
 '/Users/jrrigby1/SCIENCE/JWST_Data/LEGGOS/SGAS1402m28/Raw_data/F290LP_G395H/']

In [11]:
# Organize the output folders
folder_L2a = 'L2a/'
folder_L2b = 'L2b_pipeclean/'
folder_L3  = 'L3_destripe/'
folder_extracted = 'L3_extracted/'
output_folders = [folder_L2a, folder_L2b, folder_L3, folder_extracted]

In [12]:
# Make some dictionaries to hold files
raw_files = {}
l2a_files = {}
allmsa_calfiles = {}
allmsa_asnfile = {}

In [13]:
def sort_out_paths(thisdir, reduced_datadir, pipev='1.20.2'):  # Used several times, so generalize this 
    targname = thisdir.split('/')[-2]   # this is targname for pipev=1.20.2, and visitID for pipev=2
    input_path = thisdir  
    if pipev=='1.20.2':
        epochname =   basename(re.sub(r'\/$', '', thisdir))
        output_path = reduced_datadir + targname + '/' + epochname + '/'     # where to write result 
    elif pipev=='2.0.1':
        epochname = thisdir.split('/')[-3]   
        targname =  thisdir.split('/')[-2]   # this is visitID for pipev=2
        output_path = reduced_datadir + epochname + '/' + targname + '/' 
    return(epochname, targname, input_path, output_path)

In [14]:
if RUN_LEVEL0 :   # prep the directories we'll need
    for ii, thisdir in enumerate(dirs_to_handle) :   
        (epochname, targname, input_path, output_path) = sort_out_paths(thisdir, leggos_out, pipev=pipev)
        if os.path.exists(output_path) == False: # if folder doesn't exist
            print('   Creating folder ' + output_path)
            if pipev =='1.20.2':
                os.system('mkdir '  + leggos_out + '/' + targname) 
                os.system('mkdir '  + output_path) # creates the folder
            elif pipev == '2.0.1':                
                os.system('mkdir '  + leggos_out + '/' + epochname) 
                os.system('mkdir '  + output_path) # creates the folder
        
        for folder in output_folders :       #Make output folders if they don't already exist
            if os.path.exists(output_path + folder) == False: # if folder doesn't exist
                #print('   Creating folder ' + output_path + folder)
                os.system('mkdir ' + output_path + folder) # creates the folder

   Creating folder /Users/jrrigby1/SCIENCE/JWST_Data/LEGGOS/SGAS1402m28/Redux_v2.0.1_adaptivetrace/Raw_data/F170LP_G235H/
   Creating folder /Users/jrrigby1/SCIENCE/JWST_Data/LEGGOS/SGAS1402m28/Redux_v2.0.1_adaptivetrace/Raw_data/F290LP_G395H/


mkdir: /Users/jrrigby1/SCIENCE/JWST_Data/LEGGOS/SGAS1402m28/Redux_v2.0.1_adaptivetrace//Raw_data: File exists


In [15]:
if RUN_LEVEL1 :
    print('Starting Stage 1 reductions!')
    if pipev == '1.20.2':
        param_nested_dict = {'jump':{'expand_large_events': True}}
    elif pipev == '2.0.1':
        param_nested_dict = {'jump':{'expand_large_events': True}, 'picture_frame': {'skip': False}, \
                    'clean_flicker_noise':{'skip': False, 'autoparam': False, 'fit_method': 'fft', \
                    'background_method': None, 'n_sigma': 3, 'mask_science_regions': True, 'save_noise': False}}
        # fit_method='fft' is how you now invoke nsclean under pipeline v2
        # I have tried to set up the parameters to what nsclean expects, not sure I've done it right.  -JR
        # save_noise=True is useful for debugging, to check that 1/f noise has been removed.  But will fill your hard drive
    else : raise Exception("ERROR, do not have parameters for this pipeline version", pipev)

    for ii, thisdir in enumerate(dirs_to_handle): 
        (epochname, targname, input_path, output_path) = sort_out_paths(thisdir, leggos_out, pipev=pipev)
        print("Reducing visitID, epoch", targname, epochname, "which is dir", ii+1, "of", len(dirs_to_handle))
        rawfiles_thisepoch = glob.glob(input_path + '*nrs*_uncal.fits') #list the uncalibrated (level 1b) files.
        output_dir = output_path + folder_L2a
        outptd = [output_dir for _ in range(len(rawfiles_thisepoch))]
        list_of_dicts = [param_nested_dict for _ in range(len(rawfiles_thisepoch))]

        with Pool(cores2use) as pool:
            pool.starmap(run_jwst_det1, zip(rawfiles_thisepoch, outptd, list_of_dicts))

Starting Stage 1 reductions!
Reducing visitID, epoch F170LP_G235H Raw_data which is dir 1 of 2
Pipeline ran:  /Users/jrrigby1/SCIENCE/JWST_Data/LEGGOS/SGAS1402m28/Raw_data/F170LP_G235H/jw09359003001_02101_00009_nrs1_uncal.fits
Pipeline ran:  /Users/jrrigby1/SCIENCE/JWST_Data/LEGGOS/SGAS1402m28/Raw_data/F170LP_G235H/jw09359003001_02101_00003_nrs2_uncal.fits
Pipeline ran:  /Users/jrrigby1/SCIENCE/JWST_Data/LEGGOS/SGAS1402m28/Raw_data/F170LP_G235H/jw09359001001_02101_00007_nrs2_uncal.fits
Pipeline ran:  /Users/jrrigby1/SCIENCE/JWST_Data/LEGGOS/SGAS1402m28/Raw_data/F170LP_G235H/jw09359001001_02101_00006_nrs2_uncal.fits
Pipeline ran:  /Users/jrrigby1/SCIENCE/JWST_Data/LEGGOS/SGAS1402m28/Raw_data/F170LP_G235H/jw09359003001_02101_00008_nrs1_uncal.fits
Pipeline ran:  /Users/jrrigby1/SCIENCE/JWST_Data/LEGGOS/SGAS1402m28/Raw_data/F170LP_G235H/jw09359003001_02101_00002_nrs2_uncal.fits
Pipeline ran:  /Users/jrrigby1/SCIENCE/JWST_Data/LEGGOS/SGAS1402m28/Raw_data/F170LP_G235H/jw09359001001_02101_000

In [16]:
if RUN_LEVEL2 :
    print('Starting Stage 2 reductions!')
    if pipev == '1.20.2':
        param_nested_dict = {"nsclean":{'skip': False}, 'srctype':{'source_type': 'POINT'}, \
                            'pixel_replace':{'skip': False, 'algorithm': 'mingrad'}} # for pipeline v1.20.2

    elif pipev == '2.0.1':
        param_nested_dict = {'srctype':{'source_type': 'POINT'}}   # for pipeline v2.0.1

    else: raise Exception("ERROR, do not have parameters for this pipeline version", pipev)
    for ii, thisdir in enumerate(dirs_to_handle): 
        (epochname, targname, input_path, output_path) = sort_out_paths(thisdir, leggos_out, pipev=pipev)
        print("DEBUG", epochname, targname, input_path, output_path)
        files_to_run = glob.glob(output_path + folder_L2a + '*nrs*_rate.fits')
        print('Reducing dir', epochname, targname, 'which is dir', ii+1, 'of', len(dirs_to_handle))
        output_dir =  output_path + folder_L2b
        outptd = [output_dir for _ in range(len(files_to_run))]
        list_of_dicts = [param_nested_dict for _ in range(len(files_to_run))]

        with Pool(cores2use) as pool:
            pool.starmap(run_jwst_spec2, zip(files_to_run, outptd, list_of_dicts))

Starting Stage 2 reductions!
DEBUG Raw_data F170LP_G235H /Users/jrrigby1/SCIENCE/JWST_Data/LEGGOS/SGAS1402m28/Raw_data/F170LP_G235H/ /Users/jrrigby1/SCIENCE/JWST_Data/LEGGOS/SGAS1402m28/Redux_v2.0.1_adaptivetrace/Raw_data/F170LP_G235H/
Reducing dir Raw_data F170LP_G235H which is dir 1 of 2


2026-08-02 21:19:49,007 - CRDS - ERROR -  Error determining best reference for 'pars-targcentroidstep'  =   Unknown reference type 'pars-targcentroidstep'
2026-08-02 21:19:49,007 - CRDS - ERROR -  Error determining best reference for 'pars-targcentroidstep'  =   Unknown reference type 'pars-targcentroidstep'
2026-08-02 21:19:49,007 - CRDS - ERROR -  Error determining best reference for 'pars-targcentroidstep'  =   Unknown reference type 'pars-targcentroidstep'
2026-08-02 21:19:49,070 - CRDS - ERROR -  Error determining best reference for 'pars-targcentroidstep'  =   Unknown reference type 'pars-targcentroidstep'
2026-08-02 21:20:26,186 - CRDS - ERROR -  Error determining best reference for 'pars-targcentroidstep'  =   Unknown reference type 'pars-targcentroidstep'
2026-08-02 21:20:26,195 - CRDS - ERROR -  Error determining best reference for 'pars-targcentroidstep'  =   Unknown reference type 'pars-targcentroidstep'
2026-08-02 21:20:26,225 - CRDS - ERROR -  Error determining best refer

Pipeline ran:  /Users/jrrigby1/SCIENCE/JWST_Data/LEGGOS/SGAS1402m28/Redux_v2.0.1_adaptivetrace/Raw_data/F170LP_G235H/L2a/jw09359003001_02101_00002_nrs2_rate.fits
Pipeline ran:  /Users/jrrigby1/SCIENCE/JWST_Data/LEGGOS/SGAS1402m28/Redux_v2.0.1_adaptivetrace/Raw_data/F170LP_G235H/L2a/jw09359003001_02101_00003_nrs2_rate.fits
Pipeline ran:  /Users/jrrigby1/SCIENCE/JWST_Data/LEGGOS/SGAS1402m28/Redux_v2.0.1_adaptivetrace/Raw_data/F170LP_G235H/L2a/jw09359001001_02101_00001_nrs1_rate.fits
Pipeline ran:  /Users/jrrigby1/SCIENCE/JWST_Data/LEGGOS/SGAS1402m28/Redux_v2.0.1_adaptivetrace/Raw_data/F170LP_G235H/L2a/jw09359003001_02101_00001_nrs2_rate.fits
Pipeline ran:  /Users/jrrigby1/SCIENCE/JWST_Data/LEGGOS/SGAS1402m28/Redux_v2.0.1_adaptivetrace/Raw_data/F170LP_G235H/L2a/jw09359003001_02101_00009_nrs2_rate.fits
Pipeline ran:  /Users/jrrigby1/SCIENCE/JWST_Data/LEGGOS/SGAS1402m28/Redux_v2.0.1_adaptivetrace/Raw_data/F170LP_G235H/L2a/jw09359003001_02101_00008_nrs2_rate.fits
Pipeline ran:  /Users/jrrigb

2026-08-02 21:25:46,202 - CRDS - ERROR -  Error determining best reference for 'pars-targcentroidstep'  =   Unknown reference type 'pars-targcentroidstep'
2026-08-02 21:25:46,202 - CRDS - ERROR -  Error determining best reference for 'pars-targcentroidstep'  =   Unknown reference type 'pars-targcentroidstep'
2026-08-02 21:25:46,202 - CRDS - ERROR -  Error determining best reference for 'pars-targcentroidstep'  =   Unknown reference type 'pars-targcentroidstep'
2026-08-02 21:25:46,202 - CRDS - ERROR -  Error determining best reference for 'pars-targcentroidstep'  =   Unknown reference type 'pars-targcentroidstep'
2026-08-02 21:26:21,274 - CRDS - ERROR -  Error determining best reference for 'pars-targcentroidstep'  =   Unknown reference type 'pars-targcentroidstep'
2026-08-02 21:26:21,357 - CRDS - ERROR -  Error determining best reference for 'pars-targcentroidstep'  =   Unknown reference type 'pars-targcentroidstep'
2026-08-02 21:26:21,374 - CRDS - ERROR -  Error determining best refer

Pipeline ran:  /Users/jrrigby1/SCIENCE/JWST_Data/LEGGOS/SGAS1402m28/Redux_v2.0.1_adaptivetrace/Raw_data/F290LP_G395H/L2a/jw09359001001_02103_00005_nrs2_rate.fits
Pipeline ran:  /Users/jrrigby1/SCIENCE/JWST_Data/LEGGOS/SGAS1402m28/Redux_v2.0.1_adaptivetrace/Raw_data/F290LP_G395H/L2a/jw09359003001_04101_00005_nrs1_rate.fits
Pipeline ran:  /Users/jrrigby1/SCIENCE/JWST_Data/LEGGOS/SGAS1402m28/Redux_v2.0.1_adaptivetrace/Raw_data/F290LP_G395H/L2a/jw09359001001_02103_00004_nrs2_rate.fits
Pipeline ran:  /Users/jrrigby1/SCIENCE/JWST_Data/LEGGOS/SGAS1402m28/Redux_v2.0.1_adaptivetrace/Raw_data/F290LP_G395H/L2a/jw09359003001_04101_00007_nrs1_rate.fits
Pipeline ran:  /Users/jrrigby1/SCIENCE/JWST_Data/LEGGOS/SGAS1402m28/Redux_v2.0.1_adaptivetrace/Raw_data/F290LP_G395H/L2a/jw09359001001_02103_00007_nrs2_rate.fits
Pipeline ran:  /Users/jrrigby1/SCIENCE/JWST_Data/LEGGOS/SGAS1402m28/Redux_v2.0.1_adaptivetrace/Raw_data/F290LP_G395H/L2a/jw09359003001_04101_00006_nrs1_rate.fits
Pipeline ran:  /Users/jrrigb

In [17]:
if RUN_LEVEL3:  #   Level 3 processing for each epoch
    for ii, thisdir in enumerate(dirs_to_handle):  
        (epochname, targname, input_path, output_path) = sort_out_paths(thisdir, leggos_out, pipev=pipev)
        L3_destripedir = output_path + folder_L3
        L3_extractdir  = output_path + folder_extracted
        print("working on", epochname, targname, "which is dir", ii+1, "of", len(dirs_to_handle))
        for foo in (L3_destripedir, L3_extractdir):
            if os.path.exists(foo) == False:       # make a L3_destripe and L3_extract directory
                print('Creating folder ' + foo)
                os.system('mkdir ' + foo) # creates the folder
            # Write the associations needed to extract the spectra
        allmsa_calfiles  = glob.glob(output_path + folder_L2b + '/*cal.fits') 
        print("allmsa_calfiles has len", len(allmsa_calfiles))
        allmsa_asnfile   = L3_destripedir + 'L3asn.json'
        writel3asn(allmsa_calfiles, None, allmsa_asnfile, epochname + '_' + targname) 
        spec3 = Spec3Pipeline()
        spec3.output_dir = L3_destripedir
        spec3.outlier_detection.skip = False
        spec3.outlier_detection.ifu_second_check = True
        spec3.cube_build.coord_system = 'skyalign'  #'ifualign'
        spec3.save_results = True 
        spec3.extract_1d.skip = True
        # Turn on adaptive_trace_model (off by default)
        spec3.adaptive_trace_model.skip=False  
        #spec3.adaptive_trace_model.oversample = 3  # default
        spec3.adaptive_trace_model.maximum_core = 'half'  # default is None
        spec3.run(allmsa_asnfile)

working on Raw_data F170LP_G235H which is dir 1 of 2
allmsa_calfiles has len 36
working on Raw_data F290LP_G395H which is dir 2 of 2
allmsa_calfiles has len 36


In [18]:
output_path

'/Users/jrrigby1/SCIENCE/JWST_Data/LEGGOS/SGAS1402m28/Redux_v2.0.1_adaptivetrace/Raw_data/F290LP_G395H/'